# 03 — Constraints, Examples, and Few-Shot Learning

You own the routing behavior of an enterprise IT service desk. Establish a zero-shot baseline, implement example selection rather than memorizing framework syntax, compare six legitimate strategies, deliberately poison the nearest examples, and decide whether few-shot context earns its latency and token cost.

## Scenario, experimental question, and success criteria

Tickets route to access, billing, hardware, network, security, or software. Boundaries include VPN versus login, application licensing versus subscription billing, multilingual requests, urgent incidents, and instructions embedded in a ticket.

**Question:** Is adding more examples better than selecting a small, relevant, diverse, and correctly labelled set?

A defensible strategy improves held-out accuracy and macro F1, preserves urgent security routing, exposes selected IDs, and justifies estimated context tokens plus measured selection latency.

## Learning objectives and safety boundaries

You will implement keyword and TF-IDF/cosine primitives, similarity and maximal-marginal-relevance selection, seeded random and static baselines, held-out evaluation, an example-count curve, poisoning diagnosis, and production controls. Examples are demonstrations, not trusted business facts or authorization. The dataset is synthetic and contains no customer data.

## Environment and reproducibility

Python 3.10+ with pandas, matplotlib, and scikit-learn. The 24 training examples and 24 held-out tickets are versioned separately. Random selection uses seed 17. Offline mode is deterministic and finishes in under one minute. For live model execution, export your own `OPENAI_API_KEY`, set `PROMPT_COURSE_PROVIDER=openai`, and restart the kernel; never paste or print a key here.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from dataclasses import asdict
from pathlib import Path
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('src').resolve()))
LAB_PATH = Path('curriculum/beginner/03-constraints-examples-and-few-shot-learning/lab.py')
spec = spec_from_file_location('course03_lab', LAB_PATH)
lab = module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
examples, cases = lab.load_dataset()
print({'provider': os.getenv('PROMPT_COURSE_PROVIDER', 'mock'), 'training_examples': len(examples), 'held_out': len(cases)})
assert len(examples) == len(cases) == 24

## Architecture and internal behavior

`observed boundary → approved example pool → selector → context budget → route proposal → typed validation → held-out evaluation`

Static examples are easy to review but may not match a query. Random examples estimate how much selection matters. Similarity maximizes local relevance. Diversity selection balances query relevance against redundancy. Every dynamic selector runs only after tenant, permission, privacy, label-quality, and freshness filters in a real system.

In [ ]:
dataset_frame = pd.DataFrame([{'id': item.id, 'text': item.text, 'label': item.label, 'split': 'train'} for item in examples] + [{'id': item.id, 'text': item.input, 'label': item.expected, 'split': 'held_out', 'slice': item.slice} for item in cases])
dataset_frame.groupby(['split', 'label']).size().unstack(fill_value=0)

## Baseline — zero-shot keyword routing

The transparent baseline counts bounded queue-specific terms. It is cheap and debuggable but cannot represent paraphrase or every boundary. Measure it before adding examples so gains are attributable.

In [ ]:
baseline_rows = lab.run_strategy('zero_shot')
baseline_metrics = lab.metrics(baseline_rows)
print(baseline_metrics)
pd.DataFrame([asdict(row) for row in baseline_rows]).query('correct == False')[['case_id', 'slice', 'expected', 'predicted']]

## Inspect baseline failures

A ticket can lack exact keywords, combine two subsystems, or use another language. These are candidate `EXAMPLE`, `PROMPT`, or `MODEL` gaps. An embedded instruction to route suspicious content to billing is a `SECURITY` boundary: the request is data and cannot override the queue contract.

## Step 1 — one-shot, static, and random controls

One-shot retrieves the single nearest example. Static uses a reviewed fixed prefix. Seeded random is a control: if random examples perform as well as a selector, the selector may not justify its complexity.

In [ ]:
query = next(case.input for case in cases if case.id == 'EV-N01')
for strategy in ('one_shot', 'static', 'random'):
    chosen = lab.select_examples(strategy, query, count=4, examples=examples)
    print(strategy, [(item.id, item.label) for item in chosen])

## Step 2 — implement similarity selection

TF-IDF turns the approved example texts and query into sparse unigram/bigram vectors. Cosine similarity ranks direction rather than raw length. The implementation remains visible in `lab.py`; scikit-learn packages vectorization and linear algebra, not the selection policy or safety filters.

In [ ]:
similar = lab.select_examples('similarity', query, count=4, examples=examples)
[(item.id, item.label, item.text) for item in similar]

## Step 3 — reduce redundancy with diversity selection

Maximal marginal relevance scores a candidate by query relevance minus similarity to examples already selected. It can broaden boundary coverage, but its relevance/redundancy weights are policy choices that require held-out evaluation.

In [ ]:
diverse = lab.select_examples('diversity', query, count=4, examples=examples)
{'similarity': [(item.id, item.label) for item in similar], 'diversity': [(item.id, item.label) for item in diverse]}

## Run the controlled strategy comparison

Every strategy uses the identical held-out set and the same transparent classifier. `poisoned` deliberately rotates labels on the nearest examples to test whether high relevance plus bad labels can be worse than no examples.

In [ ]:
comparison = pd.DataFrame(lab.compare_strategies(count=4)).set_index('strategy')
comparison.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
comparison[['accuracy', 'macro_f1']].plot.bar(ax=axes[0], color=['#2F6FED', '#18A36B'])
axes[0].set_ylim(0, 1); axes[0].set_ylabel('Held-out score'); axes[0].set_title('Same 24 tickets: quality by strategy'); axes[0].grid(axis='y', alpha=.25)
comparison['mean_example_tokens_estimated'].plot.bar(ax=axes[1], color='#7A869A')
axes[1].set_ylabel('Estimated tokens'); axes[1].set_title('Example context cost'); axes[1].grid(axis='y', alpha=.25)
plt.show()

## Experimental question — are more examples better?

Vary only the number of similarity-selected examples from zero through eight. The curve exposes diminishing returns or regression; prompt length is not the optimization objective.

In [ ]:
count_curve = pd.DataFrame(lab.accuracy_by_example_count('similarity', maximum=8)).set_index('examples')
count_curve.round(4)

In [ ]:
figure, left = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
count_curve[['accuracy', 'macro_f1']].plot(marker='o', ax=left)
left.set_ylim(0, 1); left.set_ylabel('Held-out score'); left.set_title('More examples are not automatically better'); left.grid(alpha=.25)
right = left.twinx(); right.plot(count_curve.index, count_curve.mean_example_tokens_estimated, color='#C44536', marker='s', linestyle='--'); right.set_ylabel('Estimated example tokens')
plt.show()

## Slice and failure analysis

Aggregate scores can hide urgent security or multilingual failures. Inspect predictions by slice and compare the best legitimate selector with the poisoned condition before choosing a release candidate.

In [ ]:
selected_rows = pd.DataFrame([asdict(row) for row in lab.run_strategy('diversity', count=4)])
selected_rows.groupby('slice').correct.agg(['count', 'mean']).sort_values('mean')

## Optional live provider implementation

The typed `RouteResponse` and selected examples are provider-neutral. The next cell runs a deterministic fixture by default. Explicit OpenAI mode sends the same task through the Responses API and reports measured elapsed time plus provider usage when available. One call verifies integration; it does not replace the 24-case evaluation.

In [ ]:
live_case = next(case for case in cases if case.id == 'EV-N01')
provider_result = lab.run_provider_case(live_case, 'diversity', count=4)
print(provider_result.value.model_dump())
print({'mode': provider_result.response.mode, 'model': provider_result.response.model, 'elapsed_seconds': provider_result.response.elapsed_seconds, 'usage': provider_result.response.usage})

## Failure injection — high similarity, wrong labels

Poison the labels of the nearest examples without changing their text. This is realistic when a historical ticket was misrouted, a label taxonomy changed, or examples crossed a tenant boundary. The selector is working as designed; the example registry is defective.

In [ ]:
zero = lab.metrics(lab.run_strategy('zero_shot'))
poisoned = lab.metrics(lab.run_strategy('poisoned', count=4))
print({'zero_shot': zero, 'poisoned': poisoned})
assert poisoned['accuracy'] < zero['accuracy']

## Diagnose the failure and improve the right component

Use `PROMPT / CONTEXT / EXAMPLE / MODEL / SCHEMA / RETRIEVAL / TOOL / WORKFLOW / EVALUATOR / SECURITY / RUNTIME`. Here, relevant text with rotated labels is an `EXAMPLE` data-quality failure, not a prompt-wording failure. Mitigate with label review, versioning, lineage, tenant/permission filters, freshness rules, quarantine, and regression gates. Retest the unpoisoned registry rather than adding stronger instructions around bad examples.

## Technology comparison

| Approach | Strength | Limitation | Portability / production fit |
| --- | --- | --- | --- |
| Keyword/manual | transparent, cheap | weak paraphrase and multilingual recall | excellent fallback for bounded rules |
| TF-IDF + cosine | local, mature, inspectable | lexical semantics and per-query fitting cost | strong teaching/baseline option |
| Sentence-transformer embeddings | semantic and multilingual options | model/runtime/index dependency | strong when governed and benchmarked |
| Managed vector retrieval | scalable filtering/operations | vendor, cost, telemetry and privacy choices | strong at enterprise scale with controls |

Framework selection does not replace example approval, held-out evaluation, or token budgets.

## Production upgrade

| Notebook | Production |
| --- | --- |
| Local JSONL | governed example registry and separate versioned held-out suite |
| TF-IDF per query | approved embedding model/index with version tracking and cache |
| Synthetic global pool | tenant/permission/privacy/freshness filters before retrieval |
| Sequential selection | latency budget, batching, timeout, deterministic fallback |
| Printed example IDs | privacy-aware traces, drift and label-quality alerts |
| Estimated tokens | provider tokenizer/usage and hard context budget |
| Local comparison | CI gate, shadow traffic, canary and rollback |

## When not to use few-shot selection

Do not add examples when the zero-shot contract already meets the release gate, when the example pool is unreviewed or sensitive, when current evidence is missing, or when deterministic routing is more reliable. Few-shot examples demonstrate a mapping; they do not update model weights, authenticate a source, or authorize an action.

## Review questions, exercises, and advanced challenge

1. Why is seeded random selection a useful control?
2. What does macro F1 reveal that accuracy may hide?
3. Why can the nearest example be unsafe even when similarity is correct?
4. Which filters must run before tenant data enters an example index?

**Exercise 1:** add two approved boundary examples and predict the affected slices before rerunning.

**Exercise 2:** change the MMR relevance/redundancy weights and compare quality, selected IDs, tokens, and latency.

**Advanced challenge:** replace TF-IDF with an approved sentence-embedding model on the identical held-out suite. Compare quality, multilingual performance, latency, memory, portability, and operational complexity before recommending a release.

## Summary

Few-shot engineering is example-policy engineering. Establish the baseline, select from approved data, inspect what entered context, measure held-out behavior and cost, inject label failures, and keep examples only when the evidence supports the added complexity.